In [16]:
import numpy as np
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
def score_AAS():
    sol = []
    for i in range(0,len(test)):
        score = 0
        if test['stroke_density'].iloc[i] > 0.7:
            score += 2
        if test['complexity'].iloc[i] > 0.65:
            score += 2
        if bool(test['uses_gold_leaf'].iloc[i]):
            score += 1
        if bool(test['has_signature'].iloc[i]):
            score += 1
        if (test['num_colors'].iloc[i] > 65 and test['colorfulness'].iloc[i] > 0.7):
            score += 2
        if (test['contrast'].iloc[i] < 0.4 or test['brightness'].iloc[i] < 0.45 or test['brightness'].iloc[i] > 0.75):
            score -= 1
        sol.append('Autentic' if score >= 5 else 'Incert')
    return sol

subtask1_answer = score_AAS()

In [1]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

cluster_cols = ['painter_style_score', 'fake_style_score', 'stroke_density', 'complexity', 'colorfulness', 'brightness', 'contrast', 'num_colors']
X_cluster = scaler.fit_transform(pd.concat([train[cluster_cols], test[cluster_cols]], axis=0))
kmeans = KMeans(n_clusters=5, random_state=42, n_init=20).fit(X_cluster)
task2_preds = kmeans.predict(scaler.transform(test[cluster_cols]))

In [17]:
def preprocess_data(df):
    df = df.copy()
    dims = df['canvas_size'].str.split('x', expand=True).astype(float)
    df['canvas_width'] = dims[0]
    df['canvas_height'] = dims[1]
    df['canvas_area'] = df['canvas_width'] * df['canvas_height']
    df['canvas_aspect_ratio'] = df['canvas_width'] / df['canvas_height']
    df = df.drop('canvas_size',axis=1)
    return df

train_df = preprocess_data(train)
test_df = preprocess_data(test)

In [30]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(loss_function = 'MAE', learning_rate =0.35, iterations = 400, rsm = 0.95, l2_leaf_reg = 0.28, max_depth = 4, cat_features = ['brush_type','dominant_color','art_period_hint','auction_house','image_quality'])
model.fit(train_df.drop(['target_price','SampleID'],axis=1), train_df['target_price'])

0:	learn: 7978.4792036	total: 2.08ms	remaining: 832ms
1:	learn: 5710.7479846	total: 3.88ms	remaining: 772ms
2:	learn: 4634.4206889	total: 4.97ms	remaining: 658ms
3:	learn: 3749.4749122	total: 6.08ms	remaining: 602ms
4:	learn: 3287.4048653	total: 7.12ms	remaining: 562ms
5:	learn: 2841.2998155	total: 8.79ms	remaining: 577ms
6:	learn: 2650.6838852	total: 9.71ms	remaining: 545ms
7:	learn: 2318.0269156	total: 10.5ms	remaining: 514ms
8:	learn: 2125.4714394	total: 11.3ms	remaining: 489ms
9:	learn: 1961.8172922	total: 12ms	remaining: 467ms
10:	learn: 1861.3519331	total: 12.7ms	remaining: 449ms
11:	learn: 1778.8575010	total: 13.3ms	remaining: 432ms
12:	learn: 1709.9560263	total: 14ms	remaining: 417ms
13:	learn: 1597.7443732	total: 14.8ms	remaining: 408ms
14:	learn: 1524.5675371	total: 16.6ms	remaining: 425ms
15:	learn: 1503.6847087	total: 17.7ms	remaining: 424ms
16:	learn: 1454.6337663	total: 18.5ms	remaining: 417ms
17:	learn: 1395.5984639	total: 19.4ms	remaining: 412ms
18:	learn: 1374.6307294	

CatBoostRegressor(cat_features=['brush_type', 'dominant_color', 'art_period_hint', 'auction_house', 'image_quality'], iterations=400, l2_leaf_reg=0.28, learning_rate=0.35, loss_function='MAE', max_depth=4, rsm=0.95)

In [31]:
subtask3_results = model.predict(test_df.drop(['SampleID'],axis=1))

In [32]:
tr1 = pd.DataFrame({
    'SampleID':test['SampleID'],
    'subtaskID':'Task1',
    'Answer':subtask1_answer
})
tr2 = pd.DataFrame({
    'SampleID':test['SampleID'],
    'subtaskID':'Task2',
    'Answer':task2_preds
})
tr3 = pd.DataFrame({
    'SampleID':test['SampleID'],
    'subtaskID':'Task3',
    'Answer':subtask3_results
})

pd.concat([tr1,tr2,tr3]).to_csv('solution.csv',index=False)